# Custom solvers
In this example, we show how to define custom solvers. Our system
will again be silicon, because we are not very imaginative

In [1]:
using DFTK, LinearAlgebra

a = 10.26
lattice = a / 2 * [[0 1 1.];
                   [1 0 1.];
                   [1 1 0.]]
Si = ElementPsp(:Si; psp=load_psp("hgh/lda/Si-q4"))
atoms = [Si, Si]
positions =  [ones(3)/8, -ones(3)/8]

# We take very (very) crude parameters
model = model_DFT(lattice, atoms, positions; functionals=LDA())
basis = PlaneWaveBasis(model; Ecut=5, kgrid=[1, 1, 1]);

We define our custom fix-point solver: simply a damped fixed-point

In [2]:
function my_fp_solver(f, x0, info0; maxiter)
    mixing_factor = .7
    x = x0
    info = info0
    for n = 1:maxiter
        fx, info = f(x, info)
        if info.converged || info.timedout
            break
        end
        x = x + mixing_factor * (fx - x)
    end
    (; fixpoint=x, info)
end;

Note that the fixpoint map `f` operates on an auxiliary variable `info` for
state bookkeeping. Early termination criteria are flagged from inside
the function `f` using boolean flags `info.converged` and `info.timedout`.
For control over these criteria, see the `is_converged` and `maxtime`
keyword arguments of `self_consistent_field`.

Our eigenvalue solver just forms the dense matrix and diagonalizes
it explicitly (this only works for very small systems)

In [3]:
function my_eig_solver(A, X0; maxiter, tol, kwargs...)
    n = size(X0, 2)
    A = Array(A)
    E = eigen(A)
    λ = E.values[1:n]
    X = E.vectors[:, 1:n]
    (; λ, X, residual_norms=[], n_iter=0, converged=true, n_matvec=0)
end;

Finally we also define our custom mixing scheme. It will be a mixture
of simple mixing (for the first 2 steps) and than default to Kerker mixing.
In the mixing interface `δF` is $(ρ_\text{out} - ρ_\text{in})$, i.e.
the difference in density between two subsequent SCF steps and the `mix`
function returns $δρ$, which is added to $ρ_\text{in}$ to yield $ρ_\text{next}$,
the density for the next SCF step.

In [4]:
struct MyMixing
    n_simple  # Number of iterations for simple mixing
end
MyMixing() = MyMixing(2)

function DFTK.mix_density(mixing::MyMixing, basis, δF; n_iter, kwargs...)
    if n_iter <= mixing.n_simple
        return δF  # Simple mixing -> Do not modify update at all
    else
        # Use the default KerkerMixing from DFTK
        DFTK.mix_density(KerkerMixing(), basis, δF; kwargs...)
    end
end

That's it! Now we just run the SCF with these solvers

In [5]:
scfres = self_consistent_field(basis;
                               tol=1e-4,
                               solver=my_fp_solver,
                               eigensolver=my_eig_solver,
                               mixing=MyMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -7.233097858302                   -0.49    0.0    1.39s
  2   -7.249143563731       -1.79       -0.91    0.0    1.20s
  3   -7.251147725440       -2.70       -1.33    0.0   40.9ms
  4   -7.251289948847       -3.85       -1.64    0.0   51.4ms
  5   -7.251325979625       -4.44       -1.95    0.0   76.1ms
  6   -7.251335288896       -5.03       -2.25    0.0   39.6ms
  7   -7.251337788736       -5.60       -2.54    0.0   67.8ms
  8   -7.251338493253       -6.15       -2.82    0.0   40.6ms
  9   -7.251338702118       -6.68       -3.09    0.0   39.2ms
 10   -7.251338767024       -7.19       -3.36    0.0   52.5ms
 11   -7.251338788016       -7.68       -3.61    0.0   48.2ms
 12   -7.251338795023       -8.15       -3.86    0.0   38.8ms
 13   -7.251338797417       -8.62       -4.10    0.0   43.7ms


Note that the default convergence criterion is the difference in
density. When this gets below `tol`, the fixed-point solver terminates.
You can also customize this with the `is_converged` keyword argument to
`self_consistent_field`, as shown below.

## Customizing the convergence criterion
Here is an example of a defining a custom convergence criterion and specifying
it using the `is_converged` callback keyword to `self_consistent_field`.

In [6]:
function my_convergence_criterion(info)
    tol = 1e-10
    length(info.history_Etot) < 2 && return false
    ΔE = (info.history_Etot[end-1] - info.history_Etot[end])
    ΔE < tol
end

scfres2 = self_consistent_field(basis;
                                solver=my_fp_solver,
                                is_converged=my_convergence_criterion,
                                eigensolver=my_eig_solver,
                                mixing=MyMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -7.233097858302                   -0.49    0.0    125ms
  2   -7.249143563731       -1.79       -0.91    0.0    103ms
  3   -7.251147725440       -2.70       -1.33    0.0   42.9ms
  4   -7.251289948847       -3.85       -1.64    0.0   68.1ms
  5   -7.251325979625       -4.44       -1.95    0.0   39.9ms
  6   -7.251335288896       -5.03       -2.25    0.0   46.5ms
  7   -7.251337788736       -5.60       -2.54    0.0   64.9ms
  8   -7.251338493253       -6.15       -2.82    0.0   39.5ms
  9   -7.251338702118       -6.68       -3.09    0.0   63.7ms
 10   -7.251338767024       -7.19       -3.36    0.0   39.3ms
 11   -7.251338788016       -7.68       -3.61    0.0   63.0ms
 12   -7.251338795023       -8.15       -3.86    0.0   40.5ms
 13   -7.251338797417       -8.62       -4.10    0.0   44.2ms
 14   -7.251338798250       -9.08       -4.34    0.0   38.7ms
 15   -7.